# zTS — Playtype-Adjusted True Shooting

**zTS = rTS + Difficulty = player TS% − Expected TS%**

Where Expected TS% is computed from the player's Synergy playtype mix weighted
by each playtype's league-average TS%.  
A player scoring in harder environments (ISO, pick-and-roll creation) gets credit;
a player scoring in easier environments (cuts, rolls, putbacks) is adjusted down.

See `AGENTS.md` for full methodology.

In [ ]:
import sys
import pathlib

# Make src/ importable
HERE = pathlib.Path().resolve()
sys.path.insert(0, str(HERE / "src"))

import pandas as pd
import numpy as np

from load_data import load_box_scores, load_synergy_playtypes
from zts import (
    compute_player_ts,
    compute_league_ts,
    compute_league_ts_by_playtype,
    compute_playtype_shares,
    compute_expected_ts,
    compute_zts,
    PLAYTYPE_BUCKET,
)
from features import compute_all_features

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Load Data

In [ ]:
box = load_box_scores(min_year=2014, max_year=2025)
print(f"Box scores loaded: {len(box):,} player-seasons")
box.head(3)

In [ ]:
# Downloads from GitHub on first run, then uses local cache at data/raw/playtype_raw.csv
synergy = load_synergy_playtypes(min_year=2014, max_year=2025)
print(f"Synergy rows loaded: {len(synergy):,}")
print(f"Playtypes present:   {sorted(synergy['playtype'].unique())}")
print(f"Seasons present:     {sorted(synergy['Season'].unique())}")
synergy.head(3)

## 2. League Baselines

In [ ]:
league_ts = compute_league_ts(box)
print("League avg TS% by season:")
league_ts.set_index("Season")["League_TS"].to_frame()

In [ ]:
league_pt_ts = compute_league_ts_by_playtype(synergy)

# Show most recent season as a reference table
latest_year = league_pt_ts["Season"].max()
snapshot = (
    league_pt_ts[league_pt_ts["Season"] == latest_year]
    .copy()
    .assign(Bucket=lambda d: d["playtype"].map(PLAYTYPE_BUCKET))
    .sort_values("League_PT_TS")
    [["Bucket", "playtype", "League_PT_TS"]]
)
print(f"League TS% per playtype — Season {latest_year}:")
snapshot

## 3. Playtype Shares (sample check)

In [ ]:
shares = compute_playtype_shares(synergy)
print(f"Share table shape: {shares.shape}")

# Sanity check: shares should sum to ~1 per player-season
share_cols = [c for c in shares.columns if c.endswith("_share")]
shares["total_share"] = shares[share_cols].sum(axis=1)
print(f"Share total range: [{shares['total_share'].min():.3f}, {shares['total_share'].max():.3f}]")
shares.drop(columns=["total_share"], inplace=True)
shares.head(3)

## 4. Expected TS

In [ ]:
expected = compute_expected_ts(shares, league_pt_ts)
print(f"Expected TS rows: {len(expected):,}")
expected.describe()

## 5. Compute zTS (full pipeline)

In [ ]:
results = compute_zts(
    box_df=box,
    synergy_df=synergy,
    min_minutes=250,
    min_syn_poss=50,
)
print(f"zTS computed for {len(results):,} player-seasons")
results.head(10)

## 6. Spot Checks

In [ ]:
# Top zTS scorers in most recent season
latest = results[results["Season"] == results["Season"].max()].copy()
print(f"Top 15 by zTS — Season {latest['Season'].iloc[0]}:")
latest.head(15)[["Player", "Minutes", "TS_pct", "League_TS", "rTS", "ExpectedTS", "Difficulty", "zTS", "SynPoss"]]

In [ ]:
# Bottom 15 — players whose raw efficiency most overstates their role difficulty
print(f"Bottom 15 by zTS — Season {latest['Season'].iloc[0]}:")
latest.tail(15)[["Player", "Minutes", "TS_pct", "League_TS", "rTS", "ExpectedTS", "Difficulty", "zTS", "SynPoss"]]

In [ ]:
# Difficulty distribution — shows spread of role difficulty across players
print("Difficulty distribution (most recent season):")
latest["Difficulty"].describe()

In [ ]:
# rTS vs zTS correlation check — should be positively correlated but not identical
corr = latest[["rTS", "zTS"]].corr().loc["rTS", "zTS"]
print(f"rTS ↔ zTS Pearson r (most recent season): {corr:.3f}")
print("(Expected: high positive correlation, but not 1.0 — the difference is the adjustment)")

## 7. Save Output

In [ ]:
out_path = HERE / "data" / "processed" / "zts_results.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(out_path, index=False)
print(f"Saved {len(results):,} rows → {out_path}")

## 8. League TS% Per Playtype — All Seasons

Useful reference for understanding how playtype difficulty has shifted over time.

In [ ]:
pivot = league_pt_ts.pivot_table(
    index="Season", columns="playtype", values="League_PT_TS"
).round(1)
pivot

## 9. Feature Engineering

Player-season features computed alongside zTS. Drive efficiency is available via `compute_drive_efficiency()` but excluded from the main table.

### Offensive
| Feature | Description |
|---------|-------------|
| **Creation** | BPM-style box creation — scoring/playmaking generation, 3-pt adjusted |
| **Load** | Total offensive burden per 100 off poss |
| **ThreePtP** | 3-pt proficiency: logistic(FG3A) × FG3% |
| **cTOV_pct** | Controlled turnover rate: TOV_per100 / Load |
| **PasserRating** | Logistic passer score (0–11.3) from creation, layup%, AST/Load, height-adjusted |
| **TightThree_per100** | Contested 3-pt attempts per 100 (tight + very tight) |
| **RimAssists_per100** | Assists leading to rim FGM per 100 off poss |
| **Assisted2s_pct / Assisted3s_pct** | Share of 2s/3s that came off an assist |
| **PotentialAst_per100** | Passes that *would* be assists per 100 |
| **SecondaryAst_per100** | Assist-on-an-assist per 100 |
| **OffFoulsDrawn_per100** | Offensive fouls drawn per 100 |

### Defensive
| Feature | Description |
|---------|-------------|
| **RimPointsSaved** | Points saved at rim vs league-avg rim FG% |
| **C6_DFGA_per100** | Opponent FGA when player is closest defender within 6 ft |
| **C6_Diff_pct** | FG% differential as perimeter/closeout defender |
| **Deflections_per100** | Ball-hand deflections per 100 def poss |
| **ChargesDrawn_per100** | Charges drawn per 100 def poss |
| **Contested2PT/3PT_per100** | Contested shots per 100 def poss |
| **fTOV_per100** | Forced stops/turnovers (STL + Deflections + Charges) per 100 def poss |

### Rebounding splits (per 100)
`OREB_Contest`, `OREB_Uncontest`, `OREB_Defer`, `DREB_Contest`, `DREB_Uncontest`, `DREB_Defer`ensive possessions |
| **DriveFG_pct** | FG% on drives (paint efficiency indicator) |
| **DriveAst_pct** | % of drives that generate an assist (drive-and-kick quality) |
| **PotentialAst_per100** | Passes that *would* be assists if the shot went in — playmaking independent of teammate shooting |
| **SecondaryAst_per100** | Assist on the assist — ball-mover value |

In [ ]:
# box is already loaded above — features use the same DataFrame
features = compute_all_features(box, min_year=2014, max_year=2025)
print(f"Features computed for {len(features):,} player-seasons")
print(f"Columns: {list(features.columns)}\n")

print("NaN rate per feature (all seasons):")
feat_cols = [
    # Offensive creation
    "Creation", "Load", "ThreePtP", "cTOV_pct", "PasserRating",
    "TightThree_per100", "RimAssists_per100",
    "Assisted2s_pct", "Assisted3s_pct", "OffFoulsDrawn_per100",
    # Playmaking (tracking)
    "PotentialAst_per100", "SecondaryAst_per100",
    # Defensive (tracking)
    "RimPointsSaved", "C6_DFGA_per100", "C6_Diff_pct",
    # Hustle (2018+)
    "Deflections_per100", "ChargesDrawn_per100",
    "Contested2PTShots_per100", "Contested3PTShots_per100",
    "fTOV_per100",
    # Rebounding splits
    "OREB_Contest_per100", "OREB_Uncontest_per100", "OREB_Defer_per100",
    "DREB_Contest_per100", "DREB_Uncontest_per100", "DREB_Defer_per100",
]
for c in feat_cols:
    if c in features.columns:
        pct = features[c].isna().mean() * 100
        print(f"  {c:<35} {pct:5.1f}%"){pct:.1f}% missing")

In [ ]:
# Top creators / playmakers in most recent season
feat_latest = features[features["Season"] == features["Season"].max()].sort_values("Creation", ascending=False)
s = feat_latest["Season"].iloc[0]
print(f"Top 15 by Box Creation — Season {s}:")
feat_latest.head(15)[[
    "Player", "Creation", "Load", "cTOV_pct",
    "PasserRating", "PotentialAst_per100", "RimAssists_per100", "TightThree_per100",
]]

In [ ]:
# Top defenders in most recent season — rim + perimeter
rim_latest = features[features["Season"] == features["Season"].max()].dropna(subset=["RimPointsSaved"])
rim_latest = rim_latest.sort_values("RimPointsSaved", ascending=False)
print(f"Top 15 by Rim Points Saved — Season {rim_latest['Season'].iloc[0]}:")
display(rim_latest.head(15)[[
    "Player", "DFGA_per100", "Diff_pct", "RimPointsSaved",
    "C6_DFGA_per100", "C6_Diff_pct", "fTOV_per100",
]])

print(f"\nTop 15 by fTOV (Forced Stops / Turnovers) — Season {rim_latest['Season'].iloc[0]}:")
ftov_latest = features[features["Season"] == features["Season"].max()].sort_values("fTOV_per100", ascending=False)
display(ftov_latest.head(15)[[
    "Player", "fTOV_per100", "Deflections_per100", "ChargesDrawn_per100",
    "Contested2PTShots_per100", "Contested3PTShots_per100",
]])

## 10. Combine zTS + Features → Final Output

In [ ]:
# Feature columns to join onto the zTS output
feat_join_cols = [
    "PLAYER_ID", "Season", "AGE",
    # Offensive creation
    "ThreePtP", "Creation", "Load", "cTOV_pct", "PasserRating",
    # Rim offense
    "TightThree_per100", "RimAssists_per100",
    # Assisted FGs
    "Assisted2s_pct", "Assisted3s_pct",
    # Playmaking (tracking)
    "PotentialAst_per100", "SecondaryAst_per100",
    # Misc offense
    "OffFoulsDrawn_per100",
    # Rim defense
    "DFGA", "DFGA_per100", "Diff_pct", "rDiff_pct", "RimPointsSaved",
    # Perimeter defense (close-6)
    "C6_DFGA_per100", "C6_Diff_pct",
    # Hustle (2018+)
    "Deflections_per100", "ChargesDrawn_per100",
    "Contested2PTShots_per100", "Contested3PTShots_per100",
    # Forced stops
    "fTOV_per100",
    # Rebounding splits
    "OREB_Contest_per100", "OREB_Uncontest_per100", "OREB_Defer_per100",
    "DREB_Contest_per100", "DREB_Uncontest_per100", "DREB_Defer_per100",
]
feat_join = features[[c for c in feat_join_cols if c in features.columns]]

combined = results.merge(feat_join, on=["PLAYER_ID", "Season"], how="left")
print(f"Combined table: {combined.shape[0]:,} rows × {combined.shape[1]} columns")
combined.head(3)ined table: {len(combined):,} rows × {len(combined.columns)} columns")
combined.head(5)

In [ ]:
# Save standalone features table
feat_path = HERE / "data" / "processed" / "player_features.csv"
features.to_csv(feat_path, index=False)
print(f"Saved player_features.csv → {feat_path}")

# Save combined zTS + features table
combined_path = HERE / "data" / "processed" / "zts_with_features.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved zts_with_features.csv → {combined_path}")
print()
print("Output files:")
print(f"  zts_results.csv          — zTS metric only ({len(results):,} rows)")
print(f"  player_features.csv      — engineered features only ({len(features):,} rows)")
print(f"  zts_with_features.csv    — merged ({len(combined):,} rows, {len(combined.columns)} cols)")